In [1]:
import pandas as pd
from dotenv import load_dotenv
import os
from pathlib import Path

load_dotenv()
data_path = Path.cwd().parent.parent / 'data'

orders = pd.read_csv(data_path / 'olist_orders_dataset.csv')
order_items = pd.read_csv(data_path / 'olist_order_items_dataset.csv')
customers = pd.read_csv(data_path / 'olist_customers_dataset.csv')
products = pd.read_csv(data_path / 'olist_products_dataset.csv')
reviews = pd.read_csv(data_path / 'olist_order_reviews_dataset.csv')
category_translation = pd.read_csv(data_path / 'product_category_name_translation.csv')

print("Dados carregados para transformação!")

Dados carregados para transformação!


In [2]:
date_columns = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

print("Datas convertidas!")
print(orders[date_columns].dtypes)

Datas convertidas!
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [3]:
traducao_status = {
    'delivered': 'Entregue', 'shipped': 'Enviado', 'canceled': 'Cancelado',
    'unavailable': 'Indisponível', 'invoiced': 'Faturado',
    'processing': 'Em Processamento', 'created': 'Criado', 'approved': 'Aprovado'
}
orders['order_status_pt'] = orders['order_status'].map(traducao_status)
orders['ano_mes'] = orders['order_purchase_timestamp'].dt.to_period('M').astype(str)
orders['mes'] = orders['ano_mes']

print("Status traduzidos!")
print(orders['order_status_pt'].value_counts())

Status traduzidos!
order_status_pt
Entregue            96478
Enviado              1107
Cancelado             625
Indisponível          609
Faturado              314
Em Processamento      301
Criado                  5
Aprovado                2
Name: count, dtype: int64


In [4]:
traducao_categorias = {
    'health_beauty': 'Saúde e Beleza', 'watches_gifts': 'Relógios e Presentes',
    'bed_bath_table': 'Cama, Mesa e Banho', 'sports_leisure': 'Esporte e Lazer',
    'computers_accessories': 'Informática e Acessórios', 'furniture_decor': 'Móveis e Decoração',
    'housewares': 'Utilidades Domésticas', 'cool_stuff': 'Produtos Variados',
    'auto': 'Automotivo', 'toys': 'Brinquedos', 'garden_tools': 'Jardim e Ferramentas',
    'telephony': 'Telefonia', 'fashion_bags_accessories': 'Bolsas e Acessórios',
    'computers': 'Computadores', 'musical_instruments': 'Instrumentos Musicais',
    'small_appliances': 'Pequenos Eletrodomésticos', 'electronics': 'Eletrônicos',
    'books_general_interest': 'Livros',
    'construction_tools_construction': 'Ferramentas e Construção',
    'luggage_accessories': 'Malas e Acessórios',
}
category_translation['product_category_name_english_pt'] = (
    category_translation['product_category_name_english']
    .map(traducao_categorias)
    .fillna(category_translation['product_category_name_english'])
)

print("Categorias traduzidas!")

Categorias traduzidas!


In [5]:
vendas_mensais = orders.merge(
    order_items.groupby('order_id')['price'].sum().reset_index(), on='order_id'
).groupby('ano_mes')['price'].sum().reset_index()
vendas_mensais.columns = ['ano_mes', 'receita']
vendas_mensais = (vendas_mensais[vendas_mensais['ano_mes'] != '2018-09']
                  .sort_values('ano_mes').reset_index(drop=True))

print("Tabela vendas_mensais criada!")
print(vendas_mensais.tail())

Tabela vendas_mensais criada!


    ano_mes    receita
18  2018-04  996647.75
19  2018-05  996517.68
20  2018-06  865124.31
21  2018-07  895507.22
22  2018-08  854686.33
